# Day 2：MHA / MQA / GQA —— 从 KV Cache 和显存带宽理解现代大模型为什么转向 GQA

> 学习路线定位：Day 1 已经掌握 `QKᵀ / √d_k / softmax / V`。  
> Day 2 的目标不是重复 Attention 公式，而是把 **Multi-Head Attention 的 head 结构**、**KV Cache 显存公式**、**推理时显存带宽瓶颈** 建立成直觉。

---

## 今日学习大纲

| 模块 | 你要学会什么 | 对应面试/工程问题 |
|---|---|---|
| 1. MHA 基础 | 为什么 Attention 要拆成多个 head | “Multi-Head 的本质收益是什么？” |
| 2. MQA | 为什么多个 Query heads 可以共享一组 K/V | “为什么 MQA 推理快？” |
| 3. GQA | 为什么 GQA 是 MHA 与 MQA 的折中 | “现代 Llama/Qwen 为什么常用 GQA？” |
| 4. KV Cache 公式 | 精确估算不同模型的 KV Cache 显存 | “为什么 7B 长上下文也会 OOM？” |
| 5. NumPy Demo | 手写 MHA/MQA/GQA 的形状流动 | “能否画出 Q/K/V 维度变化？” |
| 6. 工程映射 | Retriever / VectorStore / Embedding 抽象 | “RAG 检索是不是模型内部记忆？” |

---

## 今日完成目标

完成本 Notebook 后，你需要能做到：

1. 画出 `Q / K / V` 在 MHA、MQA、GQA 下的维度差异。
2. 说清楚 **MHA / MQA / GQA 的本质区别主要在 K/V head 数量**，不是 Query head 数量。
3. 用公式估算 KV Cache 显存。
4. 解释为什么推理阶段常常不是算力瓶颈，而是 **反复读取 KV Cache 的显存带宽瓶颈**。
5. 回答面试官追问：为什么 GQA 主要优化推理而不是训练。
6. 把这个概念映射到 RAG：Embedding / VectorStore / Retriever 是外部检索系统，不是模型内部参数记忆。

---

## 今日建议学习节奏

| 时间 | 任务 |
|---|---|
| 20 min | 看完 MHA/MQA/GQA 原理与公式 |
| 30 min | 跑完 KV Cache 显存计算与图表 |
| 30 min | 跑完 NumPy 版 MHA/MQA/GQA demo |
| 20 min | 回答面试拷问题 |
| 30 min | 看推荐视频/文章，补直觉 |

# 0. 环境检查

本 notebook 只依赖：

```bash
conda activate cxllm
conda install -y numpy pandas matplotlib ipykernel jupyterlab
```

可选：

```bash
python -m pip install torch
```

Day 2 的核心实验用 NumPy 就可以完成，不依赖 GPU。

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib imported OK")

# 1. 从 Day 1 公式回到 Multi-Head Attention

Day 1 的单头 Scaled Dot-Product Attention 是：

$$
\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

其中：

- $Q$：Query，我当前想找什么信息。
- $K$：Key，我能提供什么索引标签。
- $V$：Value，我真正携带的信息内容。
- $QK^\top$：相关性矩阵。
- $\sqrt{d_k}$：缩放因子，防止 logits 过大导致 softmax 饱和。

Multi-Head Attention 的关键变化是：

> 不再只用一组 Q/K/V，而是把隐藏维度切成多个 head，每个 head 学不同的关系模式。

---

## 1.1 标准 MHA 的形状

假设：

```text
B = batch size
T = sequence length
d_model = hidden size
H_q = num_attention_heads
H_kv = num_key_value_heads
d_h = head_dim = d_model / H_q
```

标准 MHA 中：

```text
H_q = H_kv
```

例如：

```text
d_model = 4096
H_q = 32
H_kv = 32
d_h = 128
```

输入：

$$
X \in \mathbb{R}^{B \times T \times d_{model}}
$$

线性投影后：

$$
Q \in \mathbb{R}^{B \times T \times H_q \times d_h}
$$

$$
K \in \mathbb{R}^{B \times T \times H_{kv} \times d_h}
$$

$$
V \in \mathbb{R}^{B \times T \times H_{kv} \times d_h}
$$

标准 MHA 因为 $H_q = H_{kv}$，所以每个 Query head 都有自己独立的一组 K/V。

# 2. MHA / MQA / GQA 的本质区别

很多人会误以为 MHA、MQA、GQA 是“三种不同的 attention 公式”。这不准确。

它们的核心公式仍然是：

$$
\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}\right)V
$$

真正变化的是：

> **Query heads 有多少个，以及 Key/Value heads 有多少个。**

---

## 2.1 三者定义

| 类型 | Query heads | Key/Value heads | 特点 |
|---|---:|---:|---|
| MHA | $H_q$ | $H_q$ | 每个 Q head 独享 K/V，表达能力强，但 KV Cache 大 |
| MQA | $H_q$ | 1 | 所有 Q heads 共享一组 K/V，KV Cache 最小，但可能有质量损失 |
| GQA | $H_q$ | $1 < H_{kv} < H_q$ | 每组 Q heads 共享一组 K/V，在质量和效率之间折中 |

---

## 2.2 一个直观例子

假设：

```text
H_q = 32
d_h = 128
```

| 类型 | H_kv | 每 token 每层 K 大小 | 每 token 每层 V 大小 |
|---|---:|---:|---:|
| MHA | 32 | 32 × 128 | 32 × 128 |
| GQA | 8 | 8 × 128 | 8 × 128 |
| MQA | 1 | 1 × 128 | 1 × 128 |

这就是为什么长上下文推理时，GQA/MQA 能显著降低 KV Cache。

In [ ]:
# 画一个简单的 MHA / GQA / MQA head 结构图

fig, ax = plt.subplots(figsize=(12, 4))

systems = [
    ("MHA", 32, 32, "每个 Query head 独享一组 K/V"),
    ("GQA", 32, 8, "每 4 个 Query heads 共享一组 K/V"),
    ("MQA", 32, 1, "所有 Query heads 共享一组 K/V"),
]

for idx, (name, hq, hkv, desc) in enumerate(systems):
    y = 2 - idx
    ax.text(-1.2, y, name, fontsize=14, fontweight="bold", va="center")

    # Query heads as small vertical bars
    for i in range(hq):
        ax.add_patch(plt.Rectangle((i * 0.16, y - 0.18), 0.09, 0.36, alpha=0.65))

    # KV heads as larger blocks
    start_x = 5.8
    if hkv == 32:
        kv_to_draw = 16
        label = "K/V heads: 32"
    elif hkv == 8:
        kv_to_draw = 8
        label = "K/V heads: 8"
    else:
        kv_to_draw = 1
        label = "K/V heads: 1"

    for j in range(kv_to_draw):
        ax.add_patch(plt.Rectangle((start_x + j * 0.22, y - 0.22), 0.14, 0.44, alpha=0.65))

    ax.text(0, y + 0.33, "Query heads: 32", fontsize=9)
    ax.text(start_x, y + 0.33, label, fontsize=9)
    ax.text(9.4, y, desc, fontsize=10, va="center")

ax.set_xlim(-1.5, 13.5)
ax.set_ylim(-0.7, 2.7)
ax.axis("off")
ax.set_title("MHA / GQA / MQA 的 head 结构对比：Query heads 不变，K/V heads 被压缩", fontsize=14)
plt.show()

# 3. 为什么推理阶段会被 KV Cache 卡住？

训练阶段通常输入完整序列，可以并行计算：

```text
一次性处理 T 个 token
Q/K/V 都可以批量矩阵乘法
GPU 算力利用率较高
```

但自回归推理阶段是逐 token 生成：

```text
第 1 步生成 token_1
第 2 步生成 token_2
第 3 步生成 token_3
...
```

每一步新 token 的 Query 都要去查询历史所有 token 的 K/V：

$$
Q_{new} \cdot K_{cache}^{\top}
$$

所以历史 K/V 必须一直保存。这就是 KV Cache。

---

## 3.1 KV Cache 显存公式

对于 decoder-only LLM，KV Cache 近似为：

$$
\text{KV Cache bytes}
=
B \times 2 \times L \times T \times H_{kv} \times d_h \times s
$$

其中：

| 符号 | 含义 |
|---|---|
| $B$ | batch size |
| 2 | K 和 V 两份 |
| $L$ | Transformer 层数 |
| $T$ | 当前上下文长度 |
| $H_{kv}$ | KV heads 数量 |
| $d_h$ | 每个 head 的维度 |
| $s$ | 每个元素字节数，fp16/bf16 通常为 2 bytes |

注意关键点：

> 这里是 $H_{kv}$，不是 $H_q$。  
> 所以 GQA/MQA 能直接降低 KV Cache。

In [ ]:
def kv_cache_bytes(batch_size, num_layers, seq_len, num_kv_heads, head_dim, bytes_per_element=2):
    '''计算 decoder-only LLM 的 KV Cache 字节数。'''
    return batch_size * 2 * num_layers * seq_len * num_kv_heads * head_dim * bytes_per_element

def bytes_to_gib(n_bytes):
    return n_bytes / (1024 ** 3)

# 以一个常见 7B-ish 配置为例
config = {
    "batch_size": 1,
    "num_layers": 32,
    "seq_len": 8192,
    "num_attention_heads": 32,
    "head_dim": 128,
    "bytes_per_element": 2,  # fp16 / bf16
}

rows = []
for attn_type, h_kv in [("MHA", 32), ("GQA", 8), ("MQA", 1)]:
    n_bytes = kv_cache_bytes(
        batch_size=config["batch_size"],
        num_layers=config["num_layers"],
        seq_len=config["seq_len"],
        num_kv_heads=h_kv,
        head_dim=config["head_dim"],
        bytes_per_element=config["bytes_per_element"],
    )
    rows.append({
        "Attention 类型": attn_type,
        "H_q": config["num_attention_heads"],
        "H_kv": h_kv,
        "Seq Len": config["seq_len"],
        "KV Cache (GiB)": bytes_to_gib(n_bytes),
        "相对 MHA": h_kv / config["num_attention_heads"]
    })

df = pd.DataFrame(rows)
df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.bar(df["Attention 类型"], df["KV Cache (GiB)"])

for bar, val in zip(bars, df["KV Cache (GiB)"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{val:.2f} GiB",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_ylabel("KV Cache 显存占用（GiB）")
ax.set_title("同一模型配置下，MHA / GQA / MQA 的 KV Cache 差异")
ax.grid(axis="y", alpha=0.3)
plt.show()

## 3.2 重要观察

在上面的例子中：

```text
MHA: H_kv = 32
GQA: H_kv = 8
MQA: H_kv = 1
```

所以理论上 KV Cache 比例大致是：

```text
MHA : GQA : MQA = 32 : 8 : 1
```

也就是：

```text
GQA 的 KV Cache ≈ MHA 的 1/4
MQA 的 KV Cache ≈ MHA 的 1/32
```

这就是 Day 2 的第一条核心直觉：

> GQA/MQA 不是让模型“少算 Q”，而是让模型在推理时少存、少读 K/V。

# 4. 不同上下文长度下，KV Cache 如何增长？

KV Cache 与 sequence length 是线性关系：

$$
\text{KV Cache} \propto T
$$

因此：

```text
8K -> 32K  = 4 倍
8K -> 128K = 16 倍
```

如果 batch size 也扩大，那么显存继续线性扩大：

$$
\text{KV Cache} \propto B \times T
$$

In [ ]:
seq_lens = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
attn_settings = {
    "MHA H_kv=32": 32,
    "GQA H_kv=8": 8,
    "MQA H_kv=1": 1,
}

curve_rows = []
for label, hkv in attn_settings.items():
    for seq_len in seq_lens:
        n_bytes = kv_cache_bytes(
            batch_size=1,
            num_layers=32,
            seq_len=seq_len,
            num_kv_heads=hkv,
            head_dim=128,
            bytes_per_element=2,
        )
        curve_rows.append({
            "类型": label,
            "Seq Len": seq_len,
            "KV Cache (GiB)": bytes_to_gib(n_bytes)
        })

curve_df = pd.DataFrame(curve_rows)
curve_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for label in curve_df["类型"].unique():
    sub = curve_df[curve_df["类型"] == label]
    ax.plot(sub["Seq Len"], sub["KV Cache (GiB)"], marker="o", label=label)

ax.set_xscale("log", base=2)
ax.set_xticks(seq_lens)
ax.set_xticklabels([f"{x//1024}K" for x in seq_lens])
ax.set_xlabel("上下文长度 Seq Len")
ax.set_ylabel("KV Cache 显存占用（GiB, batch=1）")
ax.set_title("KV Cache 随上下文长度线性增长；GQA/MQA 降低斜率")
ax.grid(True, alpha=0.3)
ax.legend()

# 标注 8K -> 32K 的 4 倍增长
ax.annotate(
    "上下文 8K → 32K：KV Cache 约 4 倍",
    xy=(32768, curve_df[(curve_df["类型"]=="GQA H_kv=8") & (curve_df["Seq Len"]==32768)]["KV Cache (GiB)"].iloc[0]),
    xytext=(4096, 10),
    arrowprops=dict(arrowstyle="->", lw=1.5),
    fontsize=10
)

plt.show()

# 5. 从显存占用进一步理解“带宽瓶颈”

KV Cache 不只是“占显存”。推理时更关键的是：

> 每生成一个新 token，都要从显存中把历史 K/V 读出来参与 attention。

这会产生显存带宽压力。

粗略理解：

```text
每步 decode 要读取的 K/V 数据量
≈ 2 × L × T × H_kv × d_h × bytes
```

如果上下文 $T$ 很长，模型每生成一个 token 都要反复读取越来越长的历史 K/V。

---

## 5.1 为什么训练阶段和推理阶段瓶颈不同？

| 阶段 | 计算方式 | 常见瓶颈 |
|---|---|---|
| 训练 / Prefill | 整段序列并行计算 | 大矩阵乘法、算力吞吐 |
| Decode 推理 | 每次只生成一个 token | KV Cache 读取、显存带宽、调度 |

所以面试回答要非常明确：

> GQA 主要优化的是推理阶段，尤其是自回归 decode 阶段的 KV Cache 体积与显存带宽读取压力。

In [ ]:
# 估算每生成 1 个 token 需要读取多少 KV Cache 数据
decode_rows = []

for seq_len in [1024, 4096, 8192, 32768, 131072]:
    for label, hkv in attn_settings.items():
        bytes_per_step = kv_cache_bytes(
            batch_size=1,
            num_layers=32,
            seq_len=seq_len,
            num_kv_heads=hkv,
            head_dim=128,
            bytes_per_element=2,
        )
        decode_rows.append({
            "类型": label,
            "Seq Len": seq_len,
            "每 decode step 需读取 K/V 近似量 (GiB)": bytes_to_gib(bytes_per_step)
        })

decode_df = pd.DataFrame(decode_rows)
decode_df.pivot(index="Seq Len", columns="类型", values="每 decode step 需读取 K/V 近似量 (GiB)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

pivot = decode_df.pivot(index="Seq Len", columns="类型", values="每 decode step 需读取 K/V 近似量 (GiB)")
pivot.plot(kind="bar", ax=ax)

ax.set_ylabel("每个 decode step 读取 K/V 近似量（GiB）")
ax.set_title("长上下文下，每生成一个 token 都要反复读取历史 K/V")
ax.set_xlabel("上下文长度")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Attention 类型")
plt.xticks(rotation=0)
plt.show()

# 6. NumPy 手写 MHA / MQA / GQA 的形状流动

这一节不是训练模型，而是让你看清楚：

```text
Q heads = 多少
KV heads = 多少
GQA/MQA 如何把较少的 K/V heads 映射给更多 Q heads
```

核心规则：

```text
group_size = H_q / H_kv
```

例如：

```text
H_q = 8
H_kv = 2
group_size = 4
```

则：

```text
Q head 0,1,2,3 共享 KV head 0
Q head 4,5,6,7 共享 KV head 1
```

如果是 MQA：

```text
H_q = 8
H_kv = 1
group_size = 8
所有 Q heads 共享 KV head 0
```

In [ ]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / exp_x.sum(axis=axis, keepdims=True)

def repeat_kv_for_gqa(kv, num_q_heads):
    '''
    kv: [B, T, H_kv, d_h]
    return: [B, T, H_q, d_h]
    '''
    B, T, H_kv, d_h = kv.shape
    assert num_q_heads % H_kv == 0, "H_q 必须能被 H_kv 整除"
    repeats = num_q_heads // H_kv
    return np.repeat(kv, repeats=repeats, axis=2)

def attention_with_kv_heads(Q, K, V):
    '''
    Q: [B, T, H_q, d_h]
    K: [B, T, H_kv, d_h]
    V: [B, T, H_kv, d_h]

    如果 H_kv < H_q，先把 K/V repeat 到 H_q。
    返回:
      O: [B, T, H_q, d_h]
      P: [B, H_q, T, T]
    '''
    B, T, H_q, d_h = Q.shape
    _, _, H_kv, _ = K.shape

    if H_kv != H_q:
        K_exp = repeat_kv_for_gqa(K, H_q)
        V_exp = repeat_kv_for_gqa(V, H_q)
    else:
        K_exp = K
        V_exp = V

    # 转换成 [B, H, T, d_h] 方便做注意力
    Qh = np.transpose(Q, (0, 2, 1, 3))
    Kh = np.transpose(K_exp, (0, 2, 1, 3))
    Vh = np.transpose(V_exp, (0, 2, 1, 3))

    # scores: [B, H_q, T, T]
    scores = np.matmul(Qh, np.transpose(Kh, (0, 1, 3, 2))) / math.sqrt(d_h)
    P = softmax(scores, axis=-1)

    # Oh: [B, H_q, T, d_h]
    Oh = np.matmul(P, Vh)

    # O: [B, T, H_q, d_h]
    O = np.transpose(Oh, (0, 2, 1, 3))
    return O, P

# 构造一个 toy 输入
rng = np.random.default_rng(42)

B, T = 1, 6
H_q = 8
d_h = 4

Q = rng.normal(size=(B, T, H_q, d_h))

for name, H_kv in [("MHA", 8), ("GQA", 2), ("MQA", 1)]:
    K = rng.normal(size=(B, T, H_kv, d_h))
    V = rng.normal(size=(B, T, H_kv, d_h))
    O, P = attention_with_kv_heads(Q, K, V)
    print(f"{name}:")
    print(f"  Q shape: {Q.shape}")
    print(f"  K shape: {K.shape}")
    print(f"  V shape: {V.shape}")
    print(f"  O shape: {O.shape}")
    print(f"  Attention prob shape: {P.shape}")
    print()

## 6.1 检查 GQA 的 head 分组

下面我们显式打印：

```text
H_q = 8
H_kv = 2
```

意味着：

```text
Query heads 0,1,2,3 -> KV head 0
Query heads 4,5,6,7 -> KV head 1
```

In [ ]:
def show_gqa_mapping(num_q_heads, num_kv_heads):
    assert num_q_heads % num_kv_heads == 0
    group_size = num_q_heads // num_kv_heads
    mapping = []
    for qh in range(num_q_heads):
        kvh = qh // group_size
        mapping.append({"Query head": qh, "使用的 KV head": kvh})
    return pd.DataFrame(mapping)

mapping_df = show_gqa_mapping(num_q_heads=8, num_kv_heads=2)
mapping_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))

num_q_heads = 8
num_kv_heads = 2
group_size = num_q_heads // num_kv_heads

for qh in range(num_q_heads):
    kvh = qh // group_size
    ax.scatter(qh, 1, s=500)
    ax.text(qh, 1, f"Q{qh}", ha="center", va="center", fontsize=10)

    ax.scatter(kvh * group_size + 1.5, 0, s=700, marker="s")
    ax.text(kvh * group_size + 1.5, 0, f"KV{kvh}", ha="center", va="center", fontsize=10)

    ax.plot([qh, kvh * group_size + 1.5], [0.85, 0.2], alpha=0.5)

ax.set_xlim(-1, num_q_heads)
ax.set_ylim(-0.8, 1.6)
ax.axis("off")
ax.set_title("GQA 分组示意：多个 Query heads 共享同一个 KV head")
plt.show()

# 7. 用一个 toy 例子观察 Attention 热力图

下面我们固定同一个 Q，分别使用：

```text
MHA: H_kv = H_q
GQA: H_kv = H_q / 4
MQA: H_kv = 1
```

然后画出某个 Query head 的 attention 分布。

注意：这个 toy 例子是随机权重，不代表真实模型质量。它只帮助你看清楚 **输出形状和注意力矩阵结构**。

In [ ]:
rng = np.random.default_rng(7)

B, T = 1, 10
H_q, d_h = 8, 8
Q = rng.normal(size=(B, T, H_q, d_h))

attention_maps = {}

for name, H_kv in [("MHA", 8), ("GQA", 2), ("MQA", 1)]:
    K = rng.normal(size=(B, T, H_kv, d_h))
    V = rng.normal(size=(B, T, H_kv, d_h))
    O, P = attention_with_kv_heads(Q, K, V)
    # 取 batch=0, head=0 的 T x T attention matrix
    attention_maps[name] = P[0, 0]

for name, mat in attention_maps.items():
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    im = ax.imshow(mat, aspect="auto")
    ax.set_title(f"{name}: Head 0 Attention Map")
    ax.set_xlabel("Key / Value token index")
    ax.set_ylabel("Query token index")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="attention probability")
    plt.show()

# 8. KV Cache 计算器：你以后排障要直接会算

下面这个函数是 Day 2 的核心产出之一。

你以后看到模型配置里的：

```json
{
  "num_hidden_layers": 32,
  "num_attention_heads": 32,
  "num_key_value_heads": 8,
  "hidden_size": 4096
}
```

就应该能马上算出：

```text
head_dim = hidden_size / num_attention_heads
KV cache = B × 2 × L × T × H_kv × d_h × bytes
```

In [ ]:
def kv_cache_report(
    model_name,
    hidden_size,
    num_attention_heads,
    num_key_value_heads,
    num_hidden_layers,
    seq_lens=(4096, 8192, 32768),
    batch_size=1,
    bytes_per_element=2
):
    head_dim = hidden_size // num_attention_heads
    rows = []
    for seq_len in seq_lens:
        n_bytes = kv_cache_bytes(
            batch_size=batch_size,
            num_layers=num_hidden_layers,
            seq_len=seq_len,
            num_kv_heads=num_key_value_heads,
            head_dim=head_dim,
            bytes_per_element=bytes_per_element,
        )
        rows.append({
            "model": model_name,
            "batch_size": batch_size,
            "seq_len": seq_len,
            "hidden_size": hidden_size,
            "num_attention_heads": num_attention_heads,
            "num_key_value_heads": num_key_value_heads,
            "head_dim": head_dim,
            "dtype_bytes": bytes_per_element,
            "kv_cache_gib": bytes_to_gib(n_bytes),
        })
    return pd.DataFrame(rows)

# 三个示例配置：不是精确指定某个商用模型，只用于训练显存直觉
reports = pd.concat([
    kv_cache_report("7B-ish MHA", 4096, 32, 32, 32),
    kv_cache_report("7B-ish GQA", 4096, 32, 8, 32),
    kv_cache_report("7B-ish MQA", 4096, 32, 1, 32),
], ignore_index=True)

reports

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for model in reports["model"].unique():
    sub = reports[reports["model"] == model]
    ax.plot(sub["seq_len"], sub["kv_cache_gib"], marker="o", label=model)

ax.set_xscale("log", base=2)
ax.set_xticks([4096, 8192, 32768])
ax.set_xticklabels(["4K", "8K", "32K"])
ax.set_xlabel("上下文长度")
ax.set_ylabel("KV Cache（GiB）")
ax.set_title("KV Cache 计算器：同参数量级下，H_kv 决定长上下文显存斜率")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

# 9. 小型排障题：为什么两个 7B 模型 OOM 表现不同？

假设你有两个 7B 模型：

| 模型 | num_attention_heads | num_key_value_heads | hidden_size | layers |
|---|---:|---:|---:|---:|
| Model A | 32 | 32 | 4096 | 32 |
| Model B | 32 | 8 | 4096 | 32 |

如果其他条件相同，哪个长上下文更容易 OOM？

答案：Model A。

原因不是“参数量更大”，而是：

```text
Model A 是 MHA，H_kv = 32
Model B 是 GQA，H_kv = 8
```

所以 Model A 的 KV Cache 大约是 Model B 的 4 倍。

---

## 9.1 代码验证

In [ ]:
model_a = kv_cache_report(
    model_name="Model A: MHA",
    hidden_size=4096,
    num_attention_heads=32,
    num_key_value_heads=32,
    num_hidden_layers=32,
    seq_lens=[8192, 32768, 131072]
)

model_b = kv_cache_report(
    model_name="Model B: GQA",
    hidden_size=4096,
    num_attention_heads=32,
    num_key_value_heads=8,
    num_hidden_layers=32,
    seq_lens=[8192, 32768, 131072]
)

compare = pd.concat([model_a, model_b], ignore_index=True)
compare[["model", "seq_len", "num_key_value_heads", "head_dim", "kv_cache_gib"]]

In [ ]:
pivot = compare.pivot(index="seq_len", columns="model", values="kv_cache_gib")
pivot["A/B 倍数"] = pivot["Model A: MHA"] / pivot["Model B: GQA"]
pivot

# 10. 训练阶段 vs 推理阶段：为什么 GQA 主要优化推理？

## 10.1 训练阶段

训练时通常输入完整序列：

```text
X: [B, T, d_model]
```

所有 token 并行处理。GPU 可以做大规模矩阵乘法：

```text
Q = XW_Q
K = XW_K
V = XW_V
Attention = softmax(QKᵀ)V
```

这时瓶颈更常见是：

```text
1. 大矩阵乘法算力
2. activation 显存
3. optimizer states
4. batch size / seq length
```

## 10.2 推理 Decode 阶段

推理时每次只生成一个 token：

```text
Q_new: [B, 1, H_q, d_h]
K_cache/V_cache: [B, T, H_kv, d_h]
```

每步都要读取历史 K/V：

```text
Q_new × K_cacheᵀ
```

所以瓶颈更常见是：

```text
1. KV Cache 显存占用
2. KV Cache 读取带宽
3. 长短请求调度
4. batch 动态变化
```

---

## 10.3 面试满分表达

> GQA 减少的是 Key/Value heads 的数量，因此直接降低 KV Cache 的体积和每步 decode 读取 K/V 的带宽压力。训练阶段虽然也会少一些 K/V 投影参数和激活，但训练的主要瓶颈通常是并行矩阵乘法与 activation/optimizer 显存；GQA 的核心收益主要体现在自回归推理，尤其是长上下文 decode。

# 11. 和 RAG 工程的映射：Retriever / VectorStore / Embedding 不是模型内部记忆

Day 2 的工程映射主题是：

```text
Retriever / VectorStore / Embedding 抽象，理解检索不是模型内部记忆
```

这和 MHA/MQA/GQA 的关系是：

| 模块 | 属于模型内部？ | 作用 |
|---|---|---|
| Attention | 是 | token 在上下文窗口内部互相读取信息 |
| KV Cache | 是 | 保存当前上下文历史 token 的 K/V，用于后续生成 |
| Embedding 模型 | 通常是外部组件 | 把文档/查询编码成向量 |
| VectorStore | 外部数据库 | 存储文档向量和 metadata |
| Retriever | 外部检索策略 | 根据 query 找相关文档 chunk |
| RAG 拼接上下文 | 模型输入侧工程 | 把检索结果放入 prompt，让模型在上下文内读取 |

---

## 11.1 关键边界

很多初学者会说：

> RAG 是让模型记住我的文档。

这个说法不严谨。

更准确说法是：

> RAG 不是把知识写入模型参数，而是在推理前从外部知识库检索相关片段，并把它们作为上下文输入模型。模型通过 attention 在当前 context window 中读取这些片段。

因此：

```text
模型参数记忆 = 训练得到的内部权重
RAG 检索记忆 = 外部知识库 + 当前 prompt 上下文
KV Cache = 当前会话上下文的临时 K/V 状态
```

三者不要混为一谈。

# 12. 最小 RAG 抽象伪代码

今天不要求你真正接 LangChain，但你要能看懂工程结构。

```python
documents = load_markdown_files("./notes")

chunks = split_documents(
    documents,
    chunk_size=500,
    overlap=80
)

vectors = embedding_model.embed_documents(chunks)

vector_store.add(
    vectors=vectors,
    texts=chunks,
    metadata=[{"source": "..."}]
)

query_vector = embedding_model.embed_query(user_question)

retrieved_chunks = vector_store.search(
    query_vector,
    top_k=5
)

prompt = build_prompt(
    question=user_question,
    context=retrieved_chunks
)

answer = llm.invoke(prompt)
```

这条链路里：

```text
Embedding / VectorStore / Retriever 是外部检索系统
LLM Attention 是在 prompt 内部做 token-to-token 信息读取
KV Cache 是推理时缓存当前上下文的 K/V
```

# 13. 面试官高频拷问

## Q1：MHA、MQA、GQA 的本质区别是什么？

满分回答：

> 三者的核心区别主要是 Key/Value heads 的数量。MHA 中 Query heads 和 KV heads 数量相同，每个 Query head 都有独立 K/V；MQA 中所有 Query heads 共享一组 K/V；GQA 介于两者之间，把 Query heads 分成若干组，每组共享一组 K/V。它们的 attention 公式没有本质变化，变化的是 K/V 的存储和读取规模。

---

## Q2：为什么 GQA 能降低推理显存？

满分回答：

> Decoder-only LLM 的 KV Cache 近似为 `B × 2 × L × T × H_kv × d_h × bytes`。GQA 把 `H_kv` 从 `H_q` 降到更小的组数，因此 KV Cache 显存与每步 decode 读取 K/V 的带宽都会下降。

---

## Q3：为什么 GQA 主要优化推理，而不是训练？

满分回答：

> 训练阶段完整序列可以并行处理，主要瓶颈通常是矩阵乘法吞吐、activation 和 optimizer states。推理 decode 阶段是逐 token 生成，每个新 token 都要读取历史所有 K/V；上下文越长，KV Cache 的读取带宽越成为瓶颈。GQA 直接减少 K/V heads，所以主要优化自回归推理。

---

## Q4：MQA 为什么可能有质量损失？

满分回答：

> MQA 让所有 Query heads 共享同一组 K/V，相当于压缩了不同 head 对历史上下文的索引和内容表示。KV 表达自由度下降，可能损失某些细粒度关系。GQA 保留多个 KV groups，因此通常比 MQA 更稳，是质量和效率之间的折中。

---

## Q5：同样 7B 模型，为什么长上下文显存差很多？

满分回答：

> 参数量只说明权重规模，不直接决定 KV Cache。长上下文推理还要看 `num_hidden_layers`、`num_key_value_heads`、`head_dim`、`context length`、`dtype`、`batch size`。如果一个模型是 MHA，另一个是 GQA，即使都是 7B，长上下文 KV Cache 也可能差数倍。

---

## Q6：RAG 检索结果是模型内部记忆吗？

满分回答：

> 不是。RAG 是外部检索系统把相关文档片段放入当前 prompt，模型再通过 attention 在 context window 内读取这些片段。它不等价于把知识写入模型参数。KV Cache 也只是当前上下文的临时推理状态，不是长期知识库。

# 14. 今日练习题

请你自己完成以下练习。

---

## 练习 1：手算 KV Cache

给定模型配置：

```text
batch_size = 2
num_layers = 40
seq_len = 16384
num_attention_heads = 40
num_key_value_heads = 8
head_dim = 128
dtype = fp16
```

请手算 KV Cache 约多少 GiB。

提示：

$$
B \times 2 \times L \times T \times H_{kv} \times d_h \times bytes
$$

---

## 练习 2：画出 GQA 映射

给定：

```text
H_q = 16
H_kv = 4
```

请写出 Query head 到 KV head 的映射关系。

---

## 练习 3：回答面试问题

> 为什么 GQA 不是简单地“减少 head 数量”？

标准方向：

```text
GQA 通常保留 Query heads 数量，减少的是 K/V heads。
Query heads 仍然保持多头查询能力；
K/V heads 被分组共享，从而降低 KV Cache。
```

In [ ]:
# 练习 1 参考答案：先自己算，再运行本 cell 检查
answer_bytes = kv_cache_bytes(
    batch_size=2,
    num_layers=40,
    seq_len=16384,
    num_kv_heads=8,
    head_dim=128,
    bytes_per_element=2
)

print(f"KV Cache ≈ {bytes_to_gib(answer_bytes):.2f} GiB")

In [ ]:
# 练习 2 参考答案
show_gqa_mapping(num_q_heads=16, num_kv_heads=4)

# 15. 今日学习资源推荐

以下资源建议按优先级看：

## A. 原论文 / 技术报告

1. **Fast Transformer Decoding: One Write-Head is All You Need**  
   MQA 的核心论文。重点看它为什么说增量推理受 K/V 读取带宽限制，以及为什么共享 K/V 可以加速 decode。  
   https://arxiv.org/abs/1911.02150

2. **GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints**  
   GQA 论文。重点看 GQA 如何作为 MHA 和 MQA 的折中，为什么能接近 MHA 质量并接近 MQA 速度。  
   https://arxiv.org/abs/2305.13245

3. **Qwen2 Technical Report**  
   重点看 Grouped Query Attention 小节。它明确把 GQA 和 KV cache 推理吞吐联系起来。  
   https://arxiv.org/abs/2407.10671

## B. 可视化 / 代码学习

4. **The Illustrated Transformer — Jay Alammar**  
   适合补 Multi-Head Attention 的直观图解。  
   https://jalammar.github.io/illustrated-transformer/

5. **3Blue1Brown: Attention in transformers, step-by-step**  
   适合复习 Day 1 的 attention 直觉，再进入多头结构。  
   https://www.3blue1brown.com/lessons/attention

6. **DeepLearning.AI: How Transformer LLMs Work**  
   课程覆盖 KV cache、MQA、GQA 等现代 Transformer 改进。  
   https://www.deeplearning.ai/courses/how-transformer-llms-work/

7. **DeepLearning.AI: Attention in Transformers: Concepts and Code in PyTorch**  
   如果你想继续用 PyTorch 实现 attention，可以看这个。  
   https://www.deeplearning.ai/courses/attention-in-transformers-concepts-and-code-in-pytorch

## C. 课程视频

8. **Stanford CS25: Transformers United**  
   用于建立 Transformer 大图景，尤其适合后面衔接 RoPE、KV Cache、Serving、Agent。  
   https://web.stanford.edu/class/cs25/

# 16. 今日总结

Day 2 只抓住一句话：

> MHA / MQA / GQA 的关键差异在于 Key/Value heads 的数量；KV Cache 显存与 $H_{kv}$ 线性相关；长上下文推理时，GQA 通过减少 K/V heads 降低显存占用和带宽读取压力。

你今天必须能默写的公式：

$$
\text{KV Cache bytes}
=
B \times 2 \times L \times T \times H_{kv} \times d_h \times s
$$

你今天必须能说清楚的面试答案：

```text
GQA 不是减少 Query heads，而是减少 Key/Value heads。
推理 decode 每一步都要读取历史 K/V，所以 H_kv 越小，KV Cache 越小，显存带宽压力越小。
MQA 最省但可能损失质量，MHA 表达强但 KV Cache 大，GQA 是现代大模型常用折中。
```

---

## 晚上笔记模板

```text
今日主题：MHA / MQA / GQA
核心公式：KV Cache = B × 2 × L × T × H_kv × d_h × bytes
矩阵维度：Q [B,T,H_q,d_h]，K/V [B,T,H_kv,d_h]
显存瓶颈：decode 阶段反复读取历史 K/V
面试问答：MHA/MQA/GQA 本质区别；为什么 GQA 主要优化推理
工程映射：RAG 检索不是模型内部记忆；Retriever/VectorStore 是外部知识系统
我能从源码里对应到哪个模块：config.json 里的 num_attention_heads / num_key_value_heads
```